# Neuroselect Step 11 on Google Colab

This notebook runs the locked 3,990-span held-out language research evaluation with the four already-trained adapters. It does **not** train adapters and does not insert intended targets. Checkpoints, the pinned Hugging Face model cache, and final artifacts live on Google Drive so a disconnected runtime can resume safely.

Before running: select a GPU runtime, upload `step11-language-inputs-v1.tar.gz` to the Drive path configured below, push the Colab implementation commit, and paste its exact 40-character Git SHA into `GIT_REVISION`.

In [ ]:
# Fail before installing anything if Colab assigned an incompatible GPU.
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch

assert platform.machine() == "x86_64", platform.machine()
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime > Change runtime type > GPU"
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
memory_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}; compute capability {capability[0]}.{capability[1]}; {memory_gib:.1f} GiB")
assert capability >= (7, 5), (
    "This GPU is too old for MLX-CUDA. Reconnect until Colab assigns T4, L4, A100, "
    "or another NVIDIA GPU with compute capability >= 7.5."
)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# EDIT GIT_REVISION. Keep the other defaults unless your Drive layout differs.
REPOSITORY_URL = "https://github.com/NickSomitsch/neuroselect-bci.git"
GIT_REVISION = "PASTE_THE_40_CHARACTER_COMMIT_SHA_HERE"
DRIVE_ROOT = Path("/content/drive/MyDrive/neuroselect-step11")
BUNDLE_PATH = DRIVE_ROOT / "step11-language-inputs-v1.tar.gz"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoint-v1"
RESULT_DIR = DRIVE_ROOT / "held-out-language-personalization-research-v1"
HF_HOME = DRIVE_ROOT / "huggingface-cache"
REPOSITORY_DIR = Path("/content/neuroselect-bci-step11")

assert len(GIT_REVISION) == 40 and all(c in "0123456789abcdef" for c in GIT_REVISION), (
    "Paste the exact 40-character Git commit SHA containing the Colab implementation."
)
assert BUNDLE_PATH.is_file(), f"Upload the verified input bundle to {BUNDLE_PATH}"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
print(f"Drive workspace: {DRIVE_ROOT}")


In [ ]:
# Start from a fresh, detached, byte-exact checkout.
if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
subprocess.run(["git", "checkout", "--detach", GIT_REVISION], cwd=REPOSITORY_DIR, check=True)
resolved_revision = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_DIR, check=True, capture_output=True, text=True
).stdout.strip()
status = subprocess.run(
    ["git", "status", "--porcelain"], cwd=REPOSITORY_DIR, check=True, capture_output=True, text=True
).stdout
assert resolved_revision == GIT_REVISION and not status
print(f"Clean checkout: {resolved_revision}")


In [ ]:
# Install the locked Python 3.12 environment and MLX CUDA 12 backend.
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.10.6"], check=True)
subprocess.run(["uv", "python", "install", "3.12"], cwd=REPOSITORY_DIR, check=True)
subprocess.run(
    ["uv", "sync", "--extra", "local-language-cuda", "--no-dev", "--locked", "--python", "3.12"],
    cwd=REPOSITORY_DIR,
    check=True,
)
PYTHON = REPOSITORY_DIR / ".venv/bin/python"
RUN_ENV = {**dict(__import__("os").environ), "HF_HOME": str(HF_HOME)}
subprocess.run([str(PYTHON), "scripts/check_language_cuda.py"], cwd=REPOSITORY_DIR, env=RUN_ENV, check=True)


In [ ]:
# Verify/extract only the final adapters and corpora, then cache the exact pinned Qwen snapshot.
subprocess.run(
    [str(PYTHON), "scripts/manage_language_cloud_bundle.py", "extract", str(BUNDLE_PATH), "--destination", str(REPOSITORY_DIR)],
    cwd=REPOSITORY_DIR,
    env=RUN_ENV,
    check=True,
)
subprocess.run(
    [str(PYTHON), "scripts/cache_language_model.py", "--download"],
    cwd=REPOSITORY_DIR,
    env=RUN_ENV,
    check=True,
)


## Short pilot

Run this before the full job. It uses the development message limit but the same pinned model and all four research adapters. It confirms inference, adapter switching, memory use, and approximate per-span speed. Its output is not research evidence.

In [ ]:
PILOT_DIR = Path("/content/neuroselect-step11-pilot")
subprocess.run(
    [
        str(PYTHON), "scripts/run_held_out_language_evaluation.py",
        "--config", "configs/experiments/held_out_language_personalization.yaml",
        "--adapter-suffix=-research-v1",
        "--output", str(PILOT_DIR),
        "--progress-every", "1",
        "--download",
        "--overwrite",
    ],
    cwd=REPOSITORY_DIR,
    env=RUN_ENV,
    check=True,
)


## Full Step 11

This evaluates all 3,990 spans. Every five newly completed spans are flushed to Google Drive. Re-run this same cell after a disconnect: `--resume` verifies the complete input identity and skips every valid checkpointed span. `--overwrite` applies only to the canonical final files after all spans are present.

In [ ]:
subprocess.run(
    [
        str(PYTHON), "scripts/run_held_out_language_evaluation.py",
        "--config", "configs/experiments/held_out_language_personalization_research.yaml",
        "--adapter-suffix=-research-v1",
        "--output", str(RESULT_DIR),
        "--checkpoint-dir", str(CHECKPOINT_DIR),
        "--resume",
        "--checkpoint-every", "5",
        "--progress-every", "25",
        "--download",
        "--overwrite",
    ],
    cwd=REPOSITORY_DIR,
    env=RUN_ENV,
    check=True,
)


## Strict verification and export

The verifier checks the research protocol, all 3,990 ordered teacher-forced spans, four local adapter/corpus checksums, candidate vocabulary, clean producing commit, run identity, artifact checksums, and claim eligibility.

In [ ]:
subprocess.run(
    [str(PYTHON), "scripts/verify_language_research_evaluation.py", "--artifacts", str(RESULT_DIR)],
    cwd=REPOSITORY_DIR,
    env=RUN_ENV,
    check=True,
)

import tarfile

EXPORT_PATH = DRIVE_ROOT / "held-out-language-personalization-research-v1.tar.gz"
with tarfile.open(EXPORT_PATH, "w:gz") as archive:
    archive.add(RESULT_DIR, arcname=RESULT_DIR.name)
print(f"Verified Step 11 export: {EXPORT_PATH}")
